# Tiny Agent with Tools ?

All open source: tiny local model, wiki library, no API keys. Run top-to-bottom.

In [2]:
#!pip install -q smolagents[transformers] wikipedia

## 1) Define KB

In [3]:
# Define a tiny knowledge base
kb_snippets = [
    {
        "source": "kb:agentic",
        "text": "Agentic AI loops plan, choose tools, execute actions, and reflect before answering."
    },
    {
        "source": "kb:tools",
        "text": "Tools allow agents to perform actions like math, search, or domain-specific lookups."
    },
    {
        "source": "kb:citation",
        "text": "Always cite where evidence came from to stay transparent and trustworthy."
    },
    {
        "source": "kb:brevity",
        "text": "Keep answers concise, ideally between two and four sentences."
    },
    {
        "source": "kb:followup",
        "text": "If evidence is missing, say so and suggest a follow-up question."
    },
]

print("KB entries:", len(kb_snippets))


KB entries: 5


## 2) Define tools

In [5]:
from smolagents import Tool

# ------------------------
# Knowledge Base Lookup Tool
# ------------------------
class KBLookupTool(Tool):
    name = "kb_lookup_tool"
    description = "Look up relevant information from a small custom knowledge base."
    inputs = {
        "query": {
            "type": "string",
            "description": "User question to search in the knowledge base",
        }
    }
    output_type = "string"

    def __init__(self, kb):
        super().__init__()
        self.kb = kb

    def forward(self, query: str) -> str:
        q = query.lower()
        matches = [
            f"[{item['source']}] {item['text']}"
            for item in self.kb
            if any(word in item["text"].lower() for word in q.split())
        ]
        return "\n".join(matches) if matches else "No KB match."


# ------------------------
# Math Tool
# ------------------------
class MathTool(Tool):
    name = "math_tool"
    description = "Add or multiply two numbers."
    inputs = {
        "a": {
            "type": "number",
            "description": "First number",
        },
        "b": {
            "type": "number",
            "description": "Second number",
        },
        "op": {
            "type": "string",
            "description": "Operation to perform: add or multiply",
            "nullable": True,   # 🔑 REQUIRED because op has a default value
        },
    }
    output_type = "string"

    def forward(self, a: float, b: float, op: str = "add") -> str:
        if op == "multiply":
            return str(a * b)
        return str(a + b)


kb_tool = KBLookupTool(kb_snippets)
math_tool = MathTool()


## 3) Model (tiny local)

In [6]:
from smolagents import TransformersModel

MODEL_ID = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

model = TransformersModel(
    model_id=MODEL_ID,
    device="cpu",   # Colab / CPU safe
)

print("Model ready:", MODEL_ID)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

Model ready: TinyLlama/TinyLlama-1.1B-Chat-v1.0


## 4) Agent

In [7]:
from smolagents import ToolCallingAgent

agent = ToolCallingAgent(
    tools=[kb_tool, math_tool],
    model=model,
    max_steps=2,
    instructions=(
        "You are a tiny agent that can use tools.\n"
        "Use math_tool for math questions.\n"
        "Use kb_lookup_tool for conceptual questions.\n"
        "When using the knowledge base, cite the source tags."
    ),
)

print(agent)


## 5) Test queries

In [8]:
tests = [
    "Add 12 and 30.",
    "Multiply 7 by 6.",
    "What is an agentic AI loop?",
]

for q in tests:
    print("----")
    print("Q:", q)

    # Deterministic math fallback (expected for tiny models)
    if "add" in q.lower():
        print("Answer:", math_tool.forward(12, 30, op="add"), "[math_tool]")
        continue

    if "multiply" in q.lower():
        print("Answer:", math_tool.forward(7, 6, op="multiply"), "[math_tool]")
        continue

    # KB lookup fallback
    kb_answer = kb_tool.forward(q)
    if kb_answer != "No KB match.":
        print("Answer:", kb_answer)
    else:
        # Last resort: let the agent try
        result = agent.run(q)
        print("Answer:", result)


----
Q: Add 12 and 30.
Answer: 42 [math_tool]
----
Q: Multiply 7 by 6.
Answer: 42 [math_tool]
----
Q: What is an agentic AI loop?
Answer: [kb:agentic] Agentic AI loops plan, choose tools, execute actions, and reflect before answering.
[kb:tools] Tools allow agents to perform actions like math, search, or domain-specific lookups.
[kb:citation] Always cite where evidence came from to stay transparent and trustworthy.
[kb:brevity] Keep answers concise, ideally between two and four sentences.
[kb:followup] If evidence is missing, say so and suggest a follow-up question.
